In [1]:
import pandas as pd
import yfinance as yf
import time
import requests
import io
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

# --- Configuration ---
INPUT_CSV = "indices_with_full_names.csv"
OUTPUT_DIR = "index_constituents"
SUMMARY_REPORT_FILE = "scraping_summary_report.csv"

# --- Manual Override for truly problematic URLs ---
MANUAL_URL_OVERRIDES = {}

# --- UPGRADED: Expanded "Dictionaries" for better matching ---
TICKER_COLUMN_CANDIDATES = ['Ticker', 'Symbol', 'Ticker symbol']
COMPANY_COLUMN_CANDIDATES = ['Company', 'Name', 'Security', 'Corporation'] # Added 'Corporation'
LANDMARK_HEADINGS = ['Components', 'Constituents', 'List of companies', 'Composition'] # Added 'Composition'

def find_best_wikipedia_url(index_name, original_name, index_ticker):
    """Tries multiple strategies to find the correct Wikipedia URL."""
    if index_ticker in MANUAL_URL_OVERRIDES:
        print("  - Using manual URL override.")
        return MANUAL_URL_OVERRIDES[index_ticker]

    cleaned_name = re.sub(r'\( C \)| Inde$|\(USD\)', '', index_name).strip()
    search_queries = list(dict.fromkeys([index_name, cleaned_name, original_name]))

    for query in search_queries:
        if not query or pd.isna(query): continue
        search_url = "https://en.wikipedia.org/w/api.php"
        params = {"action": "query", "list": "search", "srsearch": query, "srlimit": 1, "format": "json"}
        try:
            response = requests.get(search_url, params=params, headers={'User-Agent': 'MyCoolTool/1.0'})
            response.raise_for_status()
            data = response.json()
            if data['query']['search']:
                page_title = data['query']['search'][0]['title']
                if query.split()[0].lower() in page_title.lower():
                    print(f"  - API search for '{query}' succeeded.")
                    return f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}"
        except Exception:
            continue
    return None

def extract_expected_count(index_name):
    """Extracts a number like 50, 100, 500 from an index name using regex."""
    match = re.search(r'\b(\d{2,4})\b', index_name)
    return int(match.group(1)) if match else None

def find_constituents_table(soup):
    """Finds the constituent table by first looking under specific headings."""
    for heading_text in LANDMARK_HEADINGS:
        span = soup.find('span', class_='mw-headline', string=re.compile(f'^{re.escape(heading_text)}', re.IGNORECASE))
        if span:
            heading_tag = span.find_parent(['h2', 'h3'])
            if heading_tag:
                table = heading_tag.find_next('table', {'class': 'wikitable'})
                if table:
                    df, company_col, ticker_col = parse_table_for_columns(table)
                    if df is not None:
                        print(f"  - Found table under '{heading_text}' heading.")
                        return df, company_col, ticker_col

    print("  - No landmark heading found. Searching all tables on page...")
    all_tables = soup.find_all('table', {'class': 'wikitable'})
    for table in all_tables:
        df, company_col, ticker_col = parse_table_for_columns(table)
        if df is not None:
            return df, company_col, ticker_col
    return None, None, None

def parse_table_for_columns(table_tag):
    """Helper function to parse a bs4 table tag and check for required columns."""
    try:
        df = pd.read_html(io.StringIO(str(table_tag)))[0]
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ['_'.join(map(str, col)).strip() for col in df.columns.values]
        
        ticker_col = next((col for col in df.columns if col in TICKER_COLUMN_CANDIDATES), None)
        company_col = next((col for col in df.columns if col in COMPANY_COLUMN_CANDIDATES), None)

        if ticker_col and company_col:
            return df, company_col, ticker_col
    except Exception:
        pass
    return None, None, None

def parse_and_verify_tickers(df, company_col, ticker_col):
    """
    UPGRADED: Extracts tickers, handles complex formats, verifies them with yfinance,
    and returns a clean DataFrame.
    """
    verified_constituents = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="  Verifying Tickers"):
        company_name = row[company_col]
        raw_ticker_text = str(row[ticker_col])
        
        # --- NEW ADVANCED PARSING LOGIC ---
        # 1. Split by semicolon for cases like "GOOG; GOOGL"
        potential_tickers = raw_ticker_text.split(';')
        
        cleaned_tickers = []
        for pt in potential_tickers:
            # 2. Split by colon for cases like "NYSE: MMM", take the last part
            parts = pt.strip().split(':')
            ticker = parts[-1].strip()
            # 3. Final cleaning of annotations
            ticker = ticker.split()[0]
            if ticker:
                cleaned_tickers.append(ticker)
        # --- END OF NEW LOGIC ---

        for ticker_symbol in cleaned_tickers:
            if not ticker_symbol or len(ticker_symbol) > 12: continue
            try:
                info = yf.Ticker(ticker_symbol).info
                if info.get('marketCap') is not None or info.get('quoteType') == 'EQUITY':
                    verified_constituents.append({"Company": company_name, "Ticker": ticker_symbol})
            except Exception:
                pass # Skip invalid tickers
            
    # Remove duplicates that may arise from multiple share classes (e.g., GOOG, GOOGL for Alphabet)
    return pd.DataFrame(verified_constituents).drop_duplicates(subset=['Ticker'])


def main():
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    master_list = pd.read_csv(INPUT_CSV)
    summary_report = []

    print("--- Starting Wikipedia Constituent Scraping (Robust Mode v2) ---")

    for _, row in master_list.iterrows():
        index_name = row['Full Index Name']
        original_name = row['Original Index Name']
        index_ticker = row['Yahoo Finance Ticker']
        
        print("\n" + "="*70)
        print(f"Processing Index: {index_name} ({index_ticker})")
        print("="*70)

        status, used_url = "Failed", "N/A"
        expected_count = extract_expected_count(index_name)
        scraped_count, discrepancy = 0, None
        
        url = find_best_wikipedia_url(index_name, original_name, index_ticker)
        if not url:
            status = "No Wikipedia Page Found"
        else:
            used_url = url
            print(f"  - Using Wikipedia URL: {url}")
            try:
                response = requests.get(url, headers={'User-Agent': 'MyCoolTool/1.0'})
                soup = BeautifulSoup(response.text, 'lxml')
                raw_df, company_col, ticker_col = find_constituents_table(soup)
                
                if raw_df is not None:
                    print(f"  - Found a potential table with columns: '{company_col}', '{ticker_col}'")
                    verified_df = parse_and_verify_tickers(raw_df, company_col, ticker_col)
                    scraped_count = len(verified_df)
                    
                    if scraped_count > 0:
                        filename = f"{index_name.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')}_constituents.csv"
                        filepath = os.path.join(OUTPUT_DIR, filename)
                        verified_df.to_csv(filepath, index=False)
                        print(f"\n  ✅ Success! Saved {scraped_count} verified constituents to '{filepath}'")
                        status = "Success"
                    else:
                        print("\n  - Found a table but could not verify any constituent tickers.")
                else:
                    print("  - Could not find a valid constituent table on the page.")

            except Exception as e:
                print(f"  - An unexpected error occurred: {e}")
        
        if expected_count is not None:
            discrepancy = expected_count - scraped_count

        summary_report.append({
            "Index Name": index_name, "Status": status, "Expected Count": expected_count or "N/A",
            "Scraped Count": scraped_count, "Discrepancy": discrepancy if discrepancy is not None else "N/A",
            "Used URL": used_url
        })

    print("\n\n" + "="*80)
    print("--- SCRAPING SUMMARY REPORT ---")
    print("="*80)
    report_df = pd.DataFrame(summary_report)
    print(report_df.to_string())
    report_df.to_csv(SUMMARY_REPORT_FILE, index=False)
    print(f"\n\n✅ Summary report saved to '{SUMMARY_REPORT_FILE}'")


if __name__ == "__main__":
    main()

--- Starting Wikipedia Constituent Scraping (Robust Mode v2) ---

Processing Index: Dow Jones Global Titans 50 Inde (^DJGT)
  - API search for 'Dow Jones Global Titans 50' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: S&P GLOBAL 100 ( C ) (^SPG100)
  - API search for 'S&P GLOBAL 100 ( C )' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/S&P_100
  - No landmark heading found. Searching all tables on page...
  - Found a potential table with columns: 'Name', 'Symbol'


  Verifying Tickers: 100%|██████████| 101/101 [08:58<00:00,  5.33s/it]



  ✅ Success! Saved 101 verified constituents to 'index_constituents\S&P_GLOBAL_100__C__constituents.csv'

Processing Index: S&P GLOBAL 1200 (^SPG1200)
  - API search for 'S&P GLOBAL 1200' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/S&P_Global_1200
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: The Global Dow (USD) (^GDOW)

Processing Index: MERVAL (^MERV)
  - API search for 'MERVAL' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/MERVAL
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: IBOVESPA (^BVSP)

Processing Index: S&P/TSX Composite index (^GSPTSE)
  - API search for 'S&P/TSX Composite index' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/S&P/TSX_Composite_Index
  - No landmark heading found. Searching all tables on page...
  - Found a poten

  Verifying Tickers: 100%|██████████| 223/223 [22:33<00:00,  6.07s/it]



  ✅ Success! Saved 106 verified constituents to 'index_constituents\S&P_TSX_Composite_index_constituents.csv'

Processing Index: S&P IPSA (^IPSA)
  - API search for 'IPSA (Chile)' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/IPSA
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: IPC MEXICO (^MXX)
  - API search for 'IPC MEXICO' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/IPC
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: CBOE Volatility Index (VIX) (^VIX)

Processing Index: Dow Jones Industrial Average (^DJI)
  - API search for 'Dow Jones Industrial Average' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average
  - No landmark heading found. Searching all tables on page...
  - Found a potential table with columns: 'Company

...


  ✅ Success! Saved 20 verified constituents to 'index_constituents\SMI_PR_constituents.csv'

Processing Index: FTSE 100 (^FTSE)
  - API search for 'FTSE 100' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/FTSE_100_Index
  - No landmark heading found. Searching all tables on page...
  - Found a potential table with columns: 'Company', 'Ticker'


  Verifying Tickers: 100%|██████████| 100/100 [01:36<00:00,  1.03it/s]



  ✅ Success! Saved 32 verified constituents to 'index_constituents\FTSE_100_constituents.csv'

Processing Index: FTSE 250 (^FTMC)
  - API search for 'FTSE 250' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/FTSE_250_Index
  - No landmark heading found. Searching all tables on page...
  - Found a potential table with columns: 'Company', 'Ticker'


  Verifying Tickers: 100%|██████████| 250/250 [03:52<00:00,  1.08it/s]



  ✅ Success! Saved 52 verified constituents to 'index_constituents\FTSE_250_constituents.csv'

Processing Index: UK FTSE All Share (^FTAS)
  - API search for 'FTSE All-Share Index (UK)' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/FTSE_All-Share_Index
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: Amex Oil Index (Energy) (^XOI)
  - API search for 'Amex Oil Index (Energy)' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/Amex_Oil_Index
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processing Index: PHLX Semiconductor (^SOX)
  - API search for 'PHLX Semiconductor' succeeded.
  - Using Wikipedia URL: https://en.wikipedia.org/wiki/PHLX_Semiconductor_Sector
  - No landmark heading found. Searching all tables on page...
  - Could not find a valid constituent table on the page.

Processin